In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

%matplotlib inline

In [ ]:
# Colab setup: mount Drive before loading result files
from google.colab import drive
drive.mount('/content/drive')

## 0. Pre-Run Backup (Run This First)

Create a timestamped backup of all existing result files in Google Drive before starting any new experiments.

In [ ]:
# Backup current results before running new experiments
from pathlib import Path
from datetime import datetime
import shutil

# Source folder containing your current results
BACKUP_SOURCE = Path('/content/drive/MyDrive/CSE495B_results_quick')

# Destination root for snapshots
BACKUP_ROOT = Path('/content/drive/MyDrive/CSE495B_pre_run_backups')
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_dir = BACKUP_ROOT / f'before_new_run_{timestamp}'
backup_dir.mkdir(parents=True, exist_ok=True)

if not BACKUP_SOURCE.exists():
    print(f'Source folder not found: {BACKUP_SOURCE}')
    print('Update BACKUP_SOURCE and rerun this cell.')
else:
    files = list(BACKUP_SOURCE.rglob('*'))
    copy_count = 0
    for src in files:
        if src.is_file():
            rel = src.relative_to(BACKUP_SOURCE)
            dst = backup_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            copy_count += 1

    print('Pre-run backup complete.')
    print(f'Backup folder: {backup_dir}')
    print(f'Files copied: {copy_count}')

    # Quick check for result artifacts
    summary_count = len(list(backup_dir.rglob('*_summary.json')))
    pred_count = len(list(backup_dir.rglob('*_predictions.json')))
    print(f'Summary files: {summary_count}')
    print(f'Prediction files: {pred_count}')

In [ ]:
# Load results (robust for Colab + Google Drive paths)
import json
from pathlib import Path
import pandas as pd

# Update this if your Drive folder name is different
DRIVE_RESULTS_ROOT = Path('/content/drive/MyDrive/CSE495B_results_quick')

# Collect all summary files recursively
summary_files = sorted(DRIVE_RESULTS_ROOT.rglob('*_summary.json'))
print(f'Found {len(summary_files)} summary files under {DRIVE_RESULTS_ROOT}')

if not summary_files:
    print('No summary files found. Check your Drive path and folder contents.')
    print('Tip: run !ls -R /content/drive/MyDrive/CSE495B_results_quick')
    df = pd.DataFrame()
else:
    rows = []
    for fp in summary_files:
        with open(fp, 'r', encoding='utf-8') as f:
            data = json.load(f)

        if 'error' in data:
            continue

        config = data.get('config', {})
        evaluation = data.get('evaluation', {})
        reasoning = evaluation.get('reasoning_metrics', {})
        hallucination = evaluation.get('hallucination_metrics', {})

        rows.append({
            'source_file': str(fp),
            'model': config.get('model_name', 'unknown'),
            'prompting': config.get('prompting_strategy', 'unknown'),
            'decoding': config.get('decoding_strategy', 'unknown'),
            'dataset': config.get('dataset_name', 'unknown'),
            'seed': config.get('seed', None),
            'max_samples': config.get('max_samples', None),
            'accuracy': evaluation.get('accuracy', 0.0),
            'exact_match': evaluation.get('exact_match', 0.0),
            'avg_reasoning_steps': reasoning.get('avg_reasoning_steps', 0.0),
            'hallucination_rate': hallucination.get('contradiction_rate', 0.0),
        })

    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} experiment results')

# Show first few rows if available
if not df.empty:
    display(df.head())
else:
    print('DataFrame is empty. Update DRIVE_RESULTS_ROOT to the correct folder and re-run this cell.')

In [ ]:
# Analysis output setup: save every graph and text output from this run
from datetime import datetime

ANALYSIS_RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
ANALYSIS_OUTPUT_DIR = Path('/content/drive/MyDrive/CSE495B_analysis_outputs') / f'analysis_run_{ANALYSIS_RUN_ID}'
ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Analysis outputs will be saved to: {ANALYSIS_OUTPUT_DIR}')


def save_fig(filename: str):
    """Save current matplotlib figure to the analysis output directory."""
    out = ANALYSIS_OUTPUT_DIR / filename
    plt.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved figure: {out.name}')


def write_text(filename: str, content: str):
    """Write text content to the analysis output directory."""
    out = ANALYSIS_OUTPUT_DIR / filename
    out.write_text(content, encoding='utf-8')
    print(f'Saved text: {out.name}')

## 1. Overall Performance Summary

In [ ]:
# Summary statistics
print("Overall Statistics:")
print(f"  Mean Accuracy: {df['accuracy'].mean():.2%}")
print(f"  Std Accuracy:  {df['accuracy'].std():.2%}")
print(f"  Best:          {df['accuracy'].max():.2%}")
print(f"  Worst:         {df['accuracy'].min():.2%}")
print()
print("Best configuration:")
best = df.loc[df['accuracy'].idxmax()]
print(f"  {best['prompting']} + {best['decoding']} on {best['dataset']}")

summary_text = "\n".join([
    "Overall Statistics",
    f"Mean Accuracy: {df['accuracy'].mean():.4f}",
    f"Std Accuracy: {df['accuracy'].std():.4f}",
    f"Best Accuracy: {df['accuracy'].max():.4f}",
    f"Worst Accuracy: {df['accuracy'].min():.4f}",
    f"Best Configuration: {best['prompting']} + {best['decoding']} on {best['dataset']}",
])
write_text('01_overall_summary.txt', summary_text)

## 2. Prompting Strategy Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by prompting
prompting_stats = df.groupby('prompting')['accuracy'].agg(['mean', 'std']).sort_values('mean', ascending=False)
ax = axes[0]
bars = ax.bar(prompting_stats.index, prompting_stats['mean'], yerr=prompting_stats['std'], capsize=5)
ax.set_xlabel('Prompting Strategy')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Prompting Strategy')
ax.set_ylim(0, 1)

# Reasoning steps by prompting
ax = axes[1]
reasoning_stats = df.groupby('prompting')['avg_reasoning_steps'].mean().sort_values(ascending=False)
ax.bar(reasoning_stats.index, reasoning_stats.values, color='coral')
ax.set_xlabel('Prompting Strategy')
ax.set_ylabel('Avg Reasoning Steps')
ax.set_title('Reasoning Depth by Prompting Strategy')

plt.tight_layout()
save_fig('02_prompting_strategy_analysis.png')
plt.show()

## 3. Decoding Strategy Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

decoding_stats = df.groupby('decoding')['accuracy'].agg(['mean', 'std']).sort_values('mean', ascending=False)
bars = ax.bar(decoding_stats.index, decoding_stats['mean'], yerr=decoding_stats['std'], capsize=5, color='steelblue')
ax.set_xlabel('Decoding Strategy')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Decoding Strategy')
ax.set_ylim(0, 1)

plt.tight_layout()
save_fig('03_decoding_strategy_analysis.png')
plt.show()

## 4. Prompting × Decoding Heatmap

In [ ]:
pivot = df.pivot_table(values='accuracy', index='prompting', columns='decoding', aggfunc='mean')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='YlGnBu', ax=ax, vmin=0, vmax=1)
ax.set_title('Accuracy Heatmap: Prompting × Decoding')
plt.tight_layout()
save_fig('04_prompting_decoding_heatmap.png')
plt.show()

## 5. Dataset Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

dataset_prompting = df.pivot_table(values='accuracy', index='dataset', columns='prompting', aggfunc='mean')
dataset_prompting.plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('Dataset')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Dataset and Prompting Strategy')
ax.legend(title='Prompting')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
save_fig('05_dataset_comparison.png')
plt.show()

## 6. Reasoning Steps vs Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for prompting in df['prompting'].unique():
    subset = df[df['prompting'] == prompting]
    ax.scatter(subset['avg_reasoning_steps'], subset['accuracy'], label=prompting, s=100, alpha=0.7)

ax.set_xlabel('Average Reasoning Steps')
ax.set_ylabel('Accuracy')
ax.set_title('Reasoning Depth vs Accuracy')
ax.legend(title='Prompting')

plt.tight_layout()
save_fig('06_reasoning_steps_vs_accuracy.png')
plt.show()

## 7. Qualitative Analysis - Sample Outputs

In [ ]:
# Qualitative analysis: show side-by-side sample outputs
from pathlib import Path
import json

qual_lines = []

if df.empty:
    msg = "DataFrame is empty. Run the load-results cell first."
    print(msg)
    qual_lines.append(msg)
else:
    # Reuse the same root path from the load-results cell.
    results_root = DRIVE_RESULTS_ROOT if 'DRIVE_RESULTS_ROOT' in globals() else Path('/content/drive/MyDrive/CSE495B_results_quick')

    prediction_files = sorted(results_root.rglob('*_predictions.json'))
    msg = f"Found {len(prediction_files)} prediction files"
    print(msg)
    qual_lines.append(msg)

    if not prediction_files:
        msg = "No *_predictions.json files found. Ensure prediction files were saved to Drive."
        print(msg)
        qual_lines.append(msg)
    else:
        def load_prediction_file(fp):
            with open(fp, 'r', encoding='utf-8') as f:
                return json.load(f)

        # Preferred pair: CoT top_p vs Direct greedy.
        preferred_pairs = [
            ('cot', 'top_p', 'direct', 'greedy'),
            ('cot', 'greedy', 'direct', 'greedy'),
            ('cot', 'top_p', 'direct', 'top_p'),
            ('cot', 'greedy', 'direct', 'top_p'),
        ]

        selected = None
        for p1, d1, p2, d2 in preferred_pairs:
            left = [p for p in prediction_files if f"_{p1}_{d1}_" in p.name]
            right = [p for p in prediction_files if f"_{p2}_{d2}_" in p.name]
            if left and right:
                selected = (left[0], right[0], f"{p1}+{d1}", f"{p2}+{d2}")
                break

        if selected is None and len(prediction_files) >= 2:
            # Fallback: compare first two available files.
            selected = (prediction_files[0], prediction_files[1], "run_1", "run_2")

        if selected is None:
            # Only one file available: still show qualitative examples.
            fp = prediction_files[0]
            preds = load_prediction_file(fp)
            msg = f"Only one prediction file is available: {fp.name}"
            print(msg)
            qual_lines.append(msg)
            msg = "Showing first 3 samples from this run:"
            print(msg)
            qual_lines.append(msg)

            n = min(len(preds), 3)
            for i in range(n):
                raw = preds[i].get('raw_output', '')
                ans = preds[i].get('final_answer', '')
                block = [
                    "=" * 80,
                    f"Sample {i+1}",
                    "-" * 80,
                    f"Final answer: {ans}",
                    "Raw output (first 700 chars):",
                    raw[:700],
                ]
                print("\n".join(block))
                print("=" * 80)
                qual_lines.extend(block)
                qual_lines.append("=" * 80)
        else:
            left_fp, right_fp, left_label, right_label = selected
            left_preds = load_prediction_file(left_fp)
            right_preds = load_prediction_file(right_fp)

            msg1 = f"Using files: {left_label} -> {left_fp.name}"
            msg2 = f"Using files: {right_label} -> {right_fp.name}"
            print(msg1)
            print(msg2)
            qual_lines.extend([msg1, msg2])

            n = min(len(left_preds), len(right_preds), 3)
            msg = f"Showing {n} side-by-side sample outputs:"
            print(msg)
            qual_lines.append(msg)

            for i in range(n):
                left_raw = left_preds[i].get('raw_output', '')
                right_raw = right_preds[i].get('raw_output', '')
                left_ans = left_preds[i].get('final_answer', '')
                right_ans = right_preds[i].get('final_answer', '')

                block = [
                    "=" * 80,
                    f"Sample {i+1}",
                    "-" * 80,
                    f"{left_label} final answer: {left_ans}",
                    f"{right_label} final answer: {right_ans}",
                    f"{left_label} raw output (first 500 chars):",
                    left_raw[:500],
                    f"{right_label} raw output (first 500 chars):",
                    right_raw[:500],
                ]
                print("\n".join(block))
                print("=" * 80)
                qual_lines.extend(block)
                qual_lines.append("=" * 80)

# Save qualitative output transcript
write_text('07_qualitative_samples.txt', "\n".join(qual_lines))

## 8. Statistical Significance Testing

In [ ]:
from scipy import stats

stats_lines = []

if df.empty:
    msg = "DataFrame is empty. Load results first."
    print(msg)
    stats_lines.append(msg)
else:
    prompting_counts = df.groupby('prompting')['accuracy'].count().sort_values(ascending=False)
    print("Samples per prompting strategy:")
    print(prompting_counts.to_string())
    stats_lines.append("Samples per prompting strategy:")
    stats_lines.append(prompting_counts.to_string())
    print()

    prompting_strategies = list(prompting_counts.index)

    print("Pairwise t-tests (Prompting Strategies):")
    print("-" * 50)
    stats_lines.append("Pairwise t-tests (Prompting Strategies):")
    stats_lines.append("-" * 50)

    printed_any = False
    for i, p1 in enumerate(prompting_strategies):
        for p2 in prompting_strategies[i + 1:]:
            acc1 = df[df['prompting'] == p1]['accuracy']
            acc2 = df[df['prompting'] == p2]['accuracy']

            n1, n2 = len(acc1), len(acc2)
            if n1 < 2 or n2 < 2:
                line = f"{p1} vs {p2}: skipped (need >=2 samples each, got {n1} and {n2})"
                print(line)
                stats_lines.append(line)
                continue

            t_stat, p_value = stats.ttest_ind(acc1, acc2, equal_var=False)
            sig = "*" if p_value < 0.05 else ""
            line = f"{p1} vs {p2}: t={t_stat:.3f}, p={p_value:.4f} {sig}"
            print(line)
            stats_lines.append(line)
            printed_any = True

    if not printed_any:
        msg1 = "No valid pairwise tests were run."
        msg2 = "Run additional seeds/configs so each prompting strategy has at least 2 results."
        print(msg1)
        print(msg2)
        stats_lines.extend([msg1, msg2])

write_text('08_statistical_significance.txt', "\n".join(stats_lines))

## 9. Export Results

In [ ]:
# Export and archive ALL results to Google Drive
from pathlib import Path
from datetime import datetime
import shutil

if df.empty:
    print("DataFrame is empty. Nothing to export. Run the load-results cell first.")
else:
    # 1) Build summary tables
    summary_table = df.groupby(['prompting', 'decoding']).agg({
        'accuracy': ['mean', 'std', 'count'],
        'avg_reasoning_steps': 'mean',
        'hallucination_rate': 'mean'
    }).round(4)

    # 2) Prepare archive folder in Drive
    drive_archive_root = Path('/content/drive/MyDrive/CSE495B_all_results_archive')
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    archive_dir = drive_archive_root / f'results_snapshot_{timestamp}'
    archive_dir.mkdir(parents=True, exist_ok=True)

    # 3) Save analysis artifacts
    summary_csv = archive_dir / 'summary_table.csv'
    full_csv = archive_dir / 'all_loaded_results.csv'
    summary_table.to_csv(summary_csv)
    df.to_csv(full_csv, index=False)

    # Also persist to run output dir used by plotting/text cells
    summary_table.to_csv(ANALYSIS_OUTPUT_DIR / '09_summary_table.csv')
    df.to_csv(ANALYSIS_OUTPUT_DIR / '09_all_loaded_results.csv', index=False)

    # Save best config as quick text note
    best = df.loc[df['accuracy'].idxmax()]
    best_text = (
        f"Best configuration\n"
        f"model: {best['model']}\n"
        f"prompting: {best['prompting']}\n"
        f"decoding: {best['decoding']}\n"
        f"dataset: {best['dataset']}\n"
        f"accuracy: {best['accuracy']:.4f}\n"
    )
    (archive_dir / 'best_configuration.txt').write_text(best_text, encoding='utf-8')
    write_text('09_best_configuration.txt', best_text)

    # 4) Copy all raw JSON result files (summary + predictions) from source root
    results_root = DRIVE_RESULTS_ROOT if 'DRIVE_RESULTS_ROOT' in globals() else Path('/content/drive/MyDrive/CSE495B_results_quick')
    raw_out_dir = archive_dir / 'raw_results'
    raw_out_dir.mkdir(parents=True, exist_ok=True)

    summary_paths = sorted(results_root.rglob('*_summary.json'))
    prediction_paths = sorted(results_root.rglob('*_predictions.json'))

    copied = 0
    for fp in summary_paths + prediction_paths:
        rel = fp.relative_to(results_root)
        dst = raw_out_dir / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(fp, dst)
        copied += 1

    # 5) Copy generated analysis outputs from this run folder
    analysis_copy_dir = archive_dir / 'analysis_outputs'
    analysis_copy_dir.mkdir(parents=True, exist_ok=True)
    for src in ANALYSIS_OUTPUT_DIR.rglob('*'):
        if src.is_file():
            rel = src.relative_to(ANALYSIS_OUTPUT_DIR)
            dst = analysis_copy_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

    print('Export complete.')
    print(f'Archive folder: {archive_dir}')
    print(f'Analysis run folder: {ANALYSIS_OUTPUT_DIR}')
    print(f'Copied raw json files: {copied}')
    print(f'Saved: {summary_csv.name}, {full_csv.name}, best_configuration.txt')

    # Display table in notebook
    summary_table